# Fieldkit Project 5: Edge Video Analytics — YOLO11 PPE & Forklift Detector Training

> **Customer**: Halden Industrie GmbH (Operations & EHS Safety Lead Jana Weber, Plant Hamburg `DE-HAM`)
> **Objective**: Fine-tune YOLO11 on merged PPE & Forklift dataset with unified classes `[person, helmet, no_helmet, vest, forklift]`.
> **Hardware Target**: Kaggle T4 / P100 GPU training; export ONNX (opset 17, dynamic batch) for real-time 4-camera 5 FPS edge inference.

In [ ]:
# 1. Environment & GPU Verification
!pip install -q ultralytics onnx onnxslim

import torch
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Dataset Setup & YAML Configuration

import yaml

dataset_yaml = {
    'path': '/kaggle/input/fieldkit-ppe-forklift-dataset',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 5,
    'names': ['person', 'helmet', 'no_helmet', 'vest', 'forklift']
}

with open('/kaggle/working/ppe_forklift.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)

print("Generated /kaggle/working/ppe_forklift.yaml:")
print(yaml.dump(dataset_yaml))


In [ ]:
# 3. Train YOLO11 Nano (yolo11n.pt) Baseline
# imgsz=640, 100 epochs with early stopping patience=20, mosaic on, hsv & blur augments
model_nano = YOLO('yolo11n.pt')

results_nano = model_nano.train(
    data='/kaggle/working/ppe_forklift.yaml',
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    project='runs/ppe',
    name='yolo11n_ppe_run',
    exist_ok=True,
    device=0
)


In [ ]:
# 4. Train YOLO11 Small (yolo11s.pt) Comparison
model_small = YOLO('yolo11s.pt')

results_small = model_small.train(
    data='/kaggle/working/ppe_forklift.yaml',
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    project='runs/ppe',
    name='yolo11s_ppe_run',
    exist_ok=True,
    device=0
)


In [ ]:
# 5. Rigorous Evaluation on Test Split
print("=== YOLO11 Nano Test Evaluation ===")
metrics_nano = model_nano.val(split='test')
print(f"mAP50:    {metrics_nano.box.map50:.4f}")
print(f"mAP50-95: {metrics_nano.box.map:.4f}")

print("\n=== YOLO11 Small Test Evaluation ===")
metrics_small = model_small.val(split='test')
print(f"mAP50:    {metrics_small.box.map50:.4f}")
print(f"mAP50-95: {metrics_small.box.map:.4f}")


In [ ]:
# 6. Export Best Model to ONNX (Opset 17, Dynamic Batching)
# Select best model between nano and small
best_model = model_nano if metrics_nano.box.map50 >= 0.85 else model_small

# Export FP32 ONNX with dynamic batching
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)

# Export FP16 ONNX for GPU acceleration
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True, half=True)
print("Exported ONNX models successfully.")


In [ ]:
# 7. Copy Artifacts to /kaggle/working/ for Download
!cp runs/ppe/yolo11n_ppe_run/weights/best.onnx /kaggle/working/yolo11n_ppe_best.onnx
!cp runs/ppe/yolo11s_ppe_run/weights/best.onnx /kaggle/working/yolo11s_ppe_best.onnx 2>/dev/null || true
!cp runs/ppe/yolo11n_ppe_run/confusion_matrix.png /kaggle/working/confusion_matrix.png 2>/dev/null || true
!cp runs/ppe/yolo11n_ppe_run/results.png /kaggle/working/training_results.png 2>/dev/null || true

print("Artifacts ready in /kaggle/working:")
!ls -lh /kaggle/working/*.onnx
